In [1]:
from tools.data_analysis_tools_struct import * calculate_WMAPE.name

SyntaxError: invalid syntax (1776678828.py, line 1)

In [1]:
import pandas as pd

results = pd.read_csv(r"C:\Users\Preet Lodaya\MLOps_Bot\Results\Stat_forecast_20120426_select_articles.csv", parse_dates=['date'])

In [ ]:
import os
PPLX_API_KEY = 
os.environ["PPLX_API_KEY"] = PPLX_API_KEY

OPENAI_API_KEY = 
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

HF_KEY =
os.environ["HF_TOKEN"] = HF_KEY

In [4]:
results.columns

Index(['intersection', 'date', 'actual_sales', 'Country', 'Store', 'Division',
       'Category', 'Product', 'forecast_es', 'forecast_sarima'],
      dtype='object')

In [3]:
def df_summary(df):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist() #[col.replace(" ", "_") for col in self.df.select_dtypes(include=['number']).columns.tolist()]
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist() #[col.replace(" ", "_") for col in self.df.select_dtypes(include=['object', 'category']).columns.tolist()]
    # print(numeric_cols, categorical_cols, df.shape)
    dataset_summary = f"DATASET: Results ({df.shape[0]} rows × {df.shape[1]} columns) Columns Names: {', '.join(df.columns.tolist())}\n\n"
    return dataset_summary

In [6]:
results_final.dtypes

NameError: name 'results_final' is not defined

In [17]:
from langchain_core.callbacks.base import BaseCallbackHandler
from typing import Dict, Any, List
from datetime import datetime

class DetailedTokenCounterCallback(BaseCallbackHandler):
    def __init__(self):
        self.token_counts = []
        self.current_call_number = 0
    
    def on_llm_end(self, response, **kwargs) -> None:
        """Called when LLM ends running."""
        self.current_call_number += 1
        
        # Extract token usage and content
        usage_metadata = None
        content = None
        
        # Navigate through the nested structure
        if hasattr(response, 'generations') and response.generations:
            for generation_list in response.generations:
                for gen in generation_list:
                    # Extract content
                    if hasattr(gen, 'text'):
                        content = gen.text
                    
                    # Extract usage_metadata from the nested message.kwargs structure
                    if hasattr(gen, 'message'):
                        msg = gen.message
                        # Check if it's a dict-like object
                        if hasattr(msg, 'get'):
                            # It's a dict
                            msg_kwargs = msg.get('kwargs', {})
                            usage_metadata = msg_kwargs.get('usage_metadata')
                            if not content:
                                content = msg_kwargs.get('content')
                        elif hasattr(msg, '__dict__'):
                            # It's an object, try to access attributes
                            if hasattr(msg, 'usage_metadata'):
                                usage_metadata = msg.usage_metadata
                            if not content and hasattr(msg, 'content'):
                                content = msg.content
                    
                    # Break after first generation (they should all have same metadata)
                    if usage_metadata:
                        break
                if usage_metadata:
                    break
        
        # If we found usage metadata, record it
        if usage_metadata:
            call_info = {
                'call_number': self.current_call_number,
                'timestamp': datetime.now().isoformat(),
                'input_tokens': usage_metadata.get('input_tokens', 0),
                'output_tokens': usage_metadata.get('output_tokens', 0),
                'total_tokens': usage_metadata.get('total_tokens', 0),
                'content': content[:200] + "..." if content and len(content) > 200 else content,  # Truncate for display
                'full_content': content,  # Store full content
                'run_id': str(kwargs.get('run_id', ''))
            }
            self.token_counts.append(call_info)
            
            # Print immediately for real-time tracking
            print(f"\n🔵 LLM Call #{self.current_call_number}")
            print(f"   Input tokens: {call_info['input_tokens']}")
            print(f"   Output tokens: {call_info['output_tokens']}")
            print(f"   Total tokens: {call_info['total_tokens']}")
            print(f"   Content preview: {call_info['content']}")
        else:
            # Debug: print structure if we can't find usage_metadata
            print(f"\n⚠️  LLM Call #{self.current_call_number} - No usage metadata found")
            if hasattr(response, 'generations'):
                print(f"   Response structure: {type(response.generations)}")
    
    def get_summary(self):
        """Get detailed summary of all calls."""
        if not self.token_counts:
            return "No LLM calls tracked"
        
        summary = "\n" + "="*60
        summary += "\n📊 TOKEN USAGE SUMMARY\n"
        summary += "="*60 + "\n"
        
        for call in self.token_counts:
            summary += f"\nCall {call['call_number']} ({call['timestamp']}):"
            summary += f"\n  Input:  {call['input_tokens']:,} tokens"
            summary += f"\n  Output: {call['output_tokens']:,} tokens"
            summary += f"\n  Total:  {call['total_tokens']:,} tokens"
            summary += f"\n  Content: {call['content']}"
        
        summary += "\n\n" + "-"*60
        summary += f"\n🎯 TOTALS ACROSS {len(self.token_counts)} LLM CALLS:"
        summary += "\n" + "-"*60
        
        total_input = sum(tc['input_tokens'] for tc in self.token_counts)
        total_output = sum(tc['output_tokens'] for tc in self.token_counts)
        total = sum(tc['total_tokens'] for tc in self.token_counts)
        
        summary += f"\n  Total Input:  {total_input:,} tokens"
        summary += f"\n  Total Output: {total_output:,} tokens"
        summary += f"\n  Grand Total:  {total:,} tokens"
        summary += "\n" + "="*60 + "\n"
        
        return summary
    
    def get_dataframe(self):
        """Get token counts as a pandas DataFrame for analysis."""
        import pandas as pd
        # Create df without full_content for cleaner display
        df_data = [{k: v for k, v in call.items() if k != 'full_content'} 
                   for call in self.token_counts]
        return pd.DataFrame(df_data)
    
    def save_to_csv(self, filename='token_usage.csv'):
        """Save token usage to CSV file."""
        import pandas as pd
        df = pd.DataFrame(self.token_counts)
        df.to_csv(filename, index=False)
        print(f"✅ Token usage saved to {filename}")


In [5]:
from langchain.tools import tool
import numpy as np

class MovingAverageForecaster:
    def __init__(self, df, id_cols, date_col, target_col,
                 forecast_horizon=4, window_length=3):
        """
        df: pandas DataFrame with time series data
        id_cols: list of columns defining unique time series
        date_col: name of datetime column
        target_col: name of target column (sales)
        forecast_horizon: number of periods to forecast
        window_length: length of moving average window
        """
        self.df = df.copy()
        self.id_cols = id_cols
        self.date_col = date_col
        self.target_col = target_col
        self.forecast_horizon = forecast_horizon
        self.window_length = window_length

        # ensure date col is datetime
        self.df[self.date_col] = pd.to_datetime(self.df[self.date_col])

    def forecast(self):
        forecasts = []

        for keys, group in self.df.groupby(self.id_cols):
            group = group.sort_values(self.date_col)
            history_values = group[self.target_col].tolist()
            last_date = group[self.date_col].max()
            freq = pd.infer_freq(group[self.date_col])

            if freq is None:
                # fallback if frequency cannot be inferred
                freq = pd.inseries_offset(group[self.date_col])

            forecast_values = []

            # Recursive moving average
            for i in range(self.forecast_horizon):
                window = history_values[-self.window_length:]  # take last 'window_length' values
                next_forecast = np.mean(window)
                forecast_values.append(next_forecast)
                history_values.append(next_forecast)  # include forecast in history for next step

            # Generate forecast dates
            forecast_dates = pd.date_range(
                start=last_date + pd.tseries.frequencies.to_offset(freq),
                periods=self.forecast_horizon,
                freq=freq
            )

            forecast_df = pd.DataFrame({
                self.date_col: forecast_dates,
                f"forecast_ma_window_{self.window_length}": forecast_values
            })

            for col, val in zip(self.id_cols, keys if isinstance(keys, tuple) else [keys]):
                forecast_df[col] = val

            forecasts.append(forecast_df)

        return pd.concat(forecasts, ignore_index=True)


@tool
def moving_average_forecast_fn(tool_input) -> str:
    """
    Generates moving average forecasts for time series data.
    Args: A single JSON string with keys:
        df_id (str): The ID of the DataFrame to use
        id_cols (list): List of columns defining unique time series
        date_col (str): Name of datetime column
        target_col (str): Name of target column to forecast
        forecast_horizon (int, optional): Number of periods to forecast. Defaults to 4.
        window_length (int, optional): Length of moving average window. Defaults to 3.
    Returns: JSON with status and new df_id containing the forecasts.
    """
    import json
    
    # Parse input
    if isinstance(tool_input, str) and tool_input.strip().startswith("{"):
        payload = json.loads(tool_input)
    elif isinstance(tool_input, dict):
        payload = tool_input
    else:
        return "Invalid input format. Expected JSON string or dict."
        
    # Extract parameters
    df_id = payload["df_id"]
    id_cols = payload["id_cols"]
    date_col = payload["date_col"] 
    target_col = payload["target_col"]
    forecast_horizon = payload.get("forecast_horizon", 4)
    window_length = payload.get("window_length", 3)

    try:
        # Get DataFrame from store
        df = df_store.get(df_id)['DataFrame']
        
        # Create forecaster instance
        forecaster = MovingAverageForecaster(
            df=df,
            id_cols=id_cols,
            date_col=date_col,
            target_col=target_col,
            forecast_horizon=forecast_horizon,
            window_length=window_length
        )
        
        # Generate forecasts
        forecast_df = forecaster.forecast()
        
        # Store results
        forecast_df_id = df_store.put(
            forecast_df,
            id=f"ma_forecast_Window_{window_length}"
        )
        
        return json.dumps({
            "status": "ok",
            "df_id": forecast_df_id,
            "description": (
                f"Generated {forecast_horizon}-period moving average forecasts "
                f"using window length {window_length}. "
                f"Results stored in DataFrame with id: {forecast_df_id}"
            )
        })
        
    except Exception as e:
        return json.dumps({
            "status": "error",
            "error": str(e),
            "traceback": traceback.format_exc()
        })


In [6]:
import pandas as pd
import numpy as np
import json

class SeasonalNaiveForecaster:
    def __init__(self, df, id_cols, date_col, target_col,
                 frequency='weeks', forecast_horizon=4, seasonality='years'):
        self.df = df.copy()
        self.id_cols = id_cols
        self.date_col = date_col
        self.target_col = target_col
        self.frequency = frequency
        self.forecast_horizon = forecast_horizon
        self.seasonality = seasonality

        # Frequency mapping for offset aliases
        self.freq_map = {
            'days': 'D',
            'weeks': 'W',
            'months': 'M',
            'quarters': 'Q',
            'years': 'Y'
        }
        if self.frequency not in self.freq_map:
            raise ValueError(f"Unsupported frequency '{self.frequency}'. Use one of {list(self.freq_map.keys())}.")
        if self.seasonality not in self.freq_map:
            raise ValueError(f"Unsupported seasonality '{self.seasonality}'. Use one of {list(self.freq_map.keys())}.")

    def _seasonal_lag(self):
        """Compute seasonal lag length based on frequency and seasonality."""
        freq_to_periods = {
            ('days', 'years'): 365,
            ('weeks', 'years'): 52,
            ('months', 'years'): 12,
            ('days', 'weeks'): 7,
            ('hours', 'days'): 24
        }
        return freq_to_periods.get((self.frequency, self.seasonality), None)

    def forecast(self):
        seasonal_lag = self._seasonal_lag()
        if seasonal_lag is None:
            raise ValueError(f"No mapping for frequency='{self.frequency}' and seasonality='{self.seasonality}'.")

        df = self.df.copy()
        df[self.date_col] = pd.to_datetime(df[self.date_col])

        forecasts = []

        for keys, group in df.groupby(self.id_cols):
            group = group.sort_values(self.date_col)
            last_date = group[self.date_col].max()

            # get last full seasonal cycle
            history = group.tail(seasonal_lag)
            if len(history) < seasonal_lag:
                raise ValueError(f"Series {keys} does not have full seasonal history.")

            # repeat seasonal pattern until forecast_horizon covered
            repeats = int(np.ceil(self.forecast_horizon / seasonal_lag))
            forecast_values = np.tile(history[self.target_col].values, repeats)[:self.forecast_horizon]

            # create forecast dates
            forecast_dates = pd.date_range(
                start=last_date + pd.tseries.frequencies.to_offset(self.freq_map[self.frequency]),
                periods=self.forecast_horizon,
                freq=self.freq_map[self.frequency]
            )

            forecast_df = pd.DataFrame({
                self.date_col: forecast_dates,
                f"forecast_snaive_{self.seasonality}": forecast_values
            })
            for col, val in zip(self.id_cols, keys if isinstance(keys, tuple) else [keys]):
                forecast_df[col] = val
            forecasts.append(forecast_df)

        return pd.concat(forecasts, ignore_index=True)


@tool
def snaive_forecast_fn(tool_input) -> str:
    """
    Generates seasonal naive forecasts for time series data.
    Args: A single JSON string with keys:
        df_id (str): The ID of the DataFrame to use
        id_cols (list): List of columns defining level of intersection
        date_col (str): Name of datetime column
        target_col (str): Name of target column to forecast
        frequency (str, optional): Frequency of the data. Defaults to 'weeks'.
        forecast_horizon (int, optional): Number of periods to forecast. Defaults to 4.
        seasonality (int, optional): Length of seasonal cycles. Defaults to frequency mapped values.
    Returns: JSON with status and new df_id containing the forecasts.
    """
    try:
        # Parse input
        if isinstance(tool_input, str) and tool_input.strip().startswith("{"):
            payload = json.loads(tool_input)
        elif isinstance(tool_input, dict):
            payload = tool_input
        else:
            return json.dumps({"status": "error", "error": "Invalid input format. Expected JSON string or dict."})

        df_id = payload["df_id"]
        id_cols = payload["id_cols"]
        date_col = payload["date_col"]
        target_col = payload["target_col"]
        frequency = payload.get("frequency", "weeks")
        forecast_horizon = int(payload.get("forecast_horizon", 4))
        seasonality = payload.get("seasonality", "years")

        # Retrieve dataframe
        df = df_store.get(df_id)['DataFrame']
    except Exception as e:
        return json.dumps({"status": "error", "error": f"Missing or invalid parameters: {e}"})

    try:
        # Create and run forecaster
        snaive = SeasonalNaiveForecaster(
            df=df,
            id_cols=id_cols,
            date_col=date_col,
            target_col=target_col,
            frequency=frequency,
            forecast_horizon=forecast_horizon,
            seasonality=seasonality
        )
        forecast_df = snaive.forecast()

        # Store results
        out_id = f"snaive_forecast_{frequency}_{seasonality}"
        stored_id = df_store.put(forecast_df, id=out_id)

        return json.dumps({
            "status": "ok",
            "df_id": stored_id,
            "description": (
                f"Generated {forecast_horizon}-period seasonal-naive forecasts "
                f"(frequency={frequency}, seasonality={seasonality}). "
                f"Results stored in DataFrame with id: {stored_id}"
            )
        })
    except Exception as e:
        return json.dumps({
            "status": "error",
            "error": str(e),
            "traceback": traceback.format_exc()
        })

In [7]:
results['month'] = results['date'].dt.to_period('M')
results['quarter'] = results['date'].dt.to_period('Q')
results['year'] = results['date'].dt.to_period('Y')
results.drop('intersection', axis=1, inplace=True)

In [8]:
# stdlib
import json
import uuid
import traceback
import sys
import os
import ast
from typing import Dict

# 3rd party
import pandas as pd
from pydantic import BaseModel, Field

# LangChain v1.2.x (corrected)
from langchain_core.globals import set_debug
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI

from langchain.agents import create_agent
from langchain_core.callbacks.base import BaseCallbackHandler
from langchain_community.callbacks import get_openai_callback
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
# from langchain.chains import LLMChain

# Community / provider-specific (optional)
from langchain_community.chat_models import ChatPerplexity

# Local
from CodeExecutor_test import run_generated_code_in_subprocess
# from langchain.chains import LLMChain

# set_debug(True)

class ReActDebugger(BaseCallbackHandler):
    def __init__(self):
        self.last_raw_output = None  # store the full LLM text

    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="")  # stream as usual

    def on_llm_end(self, response, **kwargs):
        # This is where the *complete* model output is available
        self.last_raw_output = response.generations[0][0].text
        print("\n\n[RAW OUTPUT CAPTURED]\n", self.last_raw_output)

    def on_tool_start(self, tool, input_str, **kwargs):
        print(f"\n[TOOL START] {tool.name}({input_str})")

    def on_tool_end(self, output, **kwargs):
        print(f"[TOOL END] Output: {output}")

debugger = ReActDebugger()
token_counter = DetailedTokenCounterCallback()

class DataFrameStore:
    def __init__(self):
        self._store: Dict[str, pd.DataFrame] = {}

    def put(self, df: pd.DataFrame, prefix: str = None, id: str = None, description: str = None) -> str:
        """
        Store a DataFrame with optional prefix and id.
        
        Args:
            df: DataFrame to store
            prefix: Optional string to prepend to the generated id
            id: Optional specific id to use instead of generating one
            
        Returns:
            str: The id used to store the DataFrame
        """
        if id is not None:
            df_id = id
        else:
            df_id = str(uuid.uuid4())
            if prefix is not None:
                df_id = f"{prefix}_{df_id}"
                
        self._store[df_id] = {'DataFrame':df.copy(),
                              'Summary':df_summary(df),
                              'Description': description}
        return df_id

    def get(self, df_id: str) -> pd.DataFrame:
        entry = self._store[df_id]
        return {
            "DataFrame": entry["DataFrame"].copy(deep=True),
            "Summary": entry["Summary"],
            "Description": entry["Description"]
        }

    def list_ids(self):
        return list(self._store.keys())



@tool
def get_ts_frequency_fn(df_id: str, date_col: str) -> str:
    
    """
    Determines the time series frequency of the date column using the mode statistic of time differences.
    Args:
        df_id (str): The ID of the DataFrame to use
        date_col (str): The date column name
    Returns:
        Frequency: 'Daily', 'Weekly', 'Monthly', or 'Irregular'
    """
    
    import pandas as pd
    df = df_store.get(df_id)['DataFrame']
    if df is None:
        return "No dataframe available"
    if date_col not in df.columns:
        return f"Date column '{date_col}' not found in DataFrame"
    try:
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(by=date_col)
        date_diffs = df.groupby(intersection_level)[date_col].diff().dropna()
        most_common_diff = date_diffs.mode()[0]
        if most_common_diff == pd.Timedelta(days=1):
            frequency = 'Daily'
        elif most_common_diff == pd.Timedelta(weeks=1):
            frequency = 'Weekly'
        elif most_common_diff >= pd.Timedelta(days=28) and most_common_diff <= pd.Timedelta(days=31):
            frequency = 'Monthly'
        else:
            frequency = f'Irregular (most common diff: {most_common_diff})'
        
        return json.dumps({
            "status": "ok",
            "frequency": frequency
        })
    except Exception as e:
        return json.dumps({
            "status": "error",
            "description": str(e)
        })
    
@tool
def list_dfs_fn():  
    '''
    From the df_store, this functio returns list of available DataFrame ids, their summaries and descriptions.
    '''
    df_info = {}
    for df_id, content in df_store._store.items():
        df_info[df_id] = {
            "summary": content["Summary"],
            "description": content.get("Description", "")}
    
    return json.dumps({
        "status": "ok", 
        "available_dataframes": df_info
    }, indent=2)

@tool
def calculate_accuracy_fn(
                        tool_input
                        #   df_id: str, forecast_col: str, actuals_col: str, agg: dict = None
                        ):
    """Calculates Accuracy of generated forecasts using Weighted MAPE (WMAPE) metric. 
    expects both actual and prediction values to be present in same dataframe. Stores accuracies, bias (Forecast - Actual), absolute error and actuals (sum of actuals for validation period)
    for each unique error-aggregation-level combination.
    Args:- A single JSON string with keys:
        df_id (str): The ID of the DataFrame to use
        forecast_cols (list): The list of all the forecast columns to be evaluated
        actuals_col (str): The actual values column name
        date_col (str): The date column name
        error_calculation_level (list, optional): Level at which the abs error is to be calculated. Defaults to intersection_level + [date_col].
        error_aggregation_level (list, optional): Level at which the accuracy is to be aggregated. Defaults to intersection_level.
    Output:- Saves a pandas.DataFrame with calculated metrics in the df_store.
    USAGE INSTRUCTIONS:-
        1) Use default value of error_calculation_level unless specified in explicit terms by user. 
            For eg.:- a. Query:- 'Intersection with best accuracy? calculate error at State - SKU level'. Implies -> error_calculation_level=['State','SKU']
                      b. Query:- 'Find 5 Worst performing Continent - Style comb.'. Implies -> error_aggregation_level = ['Continent','Style'] but default error_calculation_level """

    # from ui.ai_analyst_tab import ToolState
    # df = pd.read_csv(r"C:\Users\Preet Lodaya\MLOps_Bot_temp\Results\Stat_forecast_20120426.csv") #ToolState.df
    import json
    import numpy as np
    import warnings
    warnings.filterwarnings("ignore")
    
    # Handle double-encoded inputs
    if isinstance(tool_input, str) and tool_input.strip().startswith("{"):
        payload = json.loads(tool_input)
    elif isinstance(tool_input, dict):
        payload = tool_input
    else:
        return "Invalid input format. Expected JSON string or dict."
    df_id = payload["df_id"]
    forecast_cols = payload["forecast_cols"]
    actuals_col = payload["actuals_col"]
    date_col = payload["date_col"]
    error_calculation_level = payload.get("error_calculation_level", intersection_level + [date_col])
    error_aggregation_level = payload.get("error_aggregation_level", intersection_level)
    
    if df_id not in df_store.list_ids():
        return json.dumps({"status":"error","description":f"No dataframe with id :- {df_id} available"})
    else:
        df = df_store.get(df_id)['DataFrame']
        
    if forecast_cols is None or actuals_col is None:
        return "forecast_col and actuals_col required"
    if type(forecast_cols) is str:
        forecast_cols = [forecast_cols]
    if not all([True if col in df.columns else False for col in forecast_cols]) or actuals_col not in df.columns:
        return json.dumps({
            "status": "error", 
            "description": f"Required columns {forecast_cols} or {actuals_col} not found in DataFrame"
        })
    try:
        # Find intersection between error aggregation and calculation levels
        abs_error_cols = [f'Absolute_Error_{f}' for f in forecast_cols]
        bias_cols = [f'Bias_{f}' for f in forecast_cols]
        accuracy_cols = [f'Accuracy_{f}' for f in forecast_cols]
        grouped_df = df.groupby(list(set(error_aggregation_level).union(set(error_calculation_level))),as_index=False)[forecast_cols + [actuals_col]].sum()
        grouped_df[bias_cols] = grouped_df[forecast_cols].subtract(grouped_df[actuals_col], axis = 0)
        grouped_df[abs_error_cols] = grouped_df[bias_cols].abs()
        
        grouped_acc = grouped_df.groupby(error_aggregation_level,as_index=False).apply(
            lambda x: 100 - (x[abs_error_cols].sum()*100 / x[actuals_col].sum())).rename(columns={f'Absolute_Error_{c}':f'Accuracy_{c}' for c in forecast_cols})
        grouped_errors = grouped_df.groupby(error_aggregation_level,as_index=False)[bias_cols + abs_error_cols + [actuals_col]].sum()#.rename(columns={f'Bias_{c}':f'Accuracy_{c}' for c in forecast_cols})
        
        grouped = pd.merge(grouped_acc, grouped_errors, on=error_aggregation_level, how='inner')
        
        id = f"Accuracy_{'_'.join(error_aggregation_level)}"
        description = f"Accuracy numbers stored in DataFrame with id:- {id}. Absolute errors calculated at {'-'.join(error_calculation_level)} and subsequently aggregated at {'-'.join(error_aggregation_level)} with relevant columns {','.join(accuracy_cols + bias_cols + abs_error_cols + [actuals_col])}."
        grouped_id = df_store.put(grouped, id=id, description=description)
        
        return json.dumps({
            "status": "ok",
            "df_id": grouped_id,
            "description": description
        })

    except Exception as e:
        # return f"WMAPE calculation error: {str(e)}"
        return json.dumps({
            "status": "Error",
            "description": str(e)
        })

# -----------------------------
# 3) Code generator tool (simple wrapper that calls an LLM to produce python code only)
#    - expects input JSON: {"goal": "what to do with df", "df_id": "..."}
#    - returns JSON: {"status":"ok","code":"<python code as string>"}
# -----------------------------

# create a small prompting function: ask LLM to return ONLY a python function named 'transform(df)'
# that accepts `df` (pandas) and returns either a dataframe or prints results.

@tool
def code_generator_tool_fn(tool_input) -> str:
    '''
    Generates Python code to operate ONLY on a dict `dfs: Dict[str, pd.DataFrame]`.
    Always stores the generated code in `global_code`. Includes optional error context for debugging failed attempts.
    Args: A single JSON string with keys:
        goal (str): What the code should achieve
        df_ids (list): The IDs of the DataFrames to use
        error_context (str, optional): Entire traceback from the error encountered from previous run for this/related goal
        store_df (bool, optional): 
            If True:- The code would produce a final DataFrame to be stored in df_store.
            If False:- The code would return a string answering the goal.
            Defaults to False
    Returns: A JSON string with keys:
        status (str): "ok" or "error" 
        Description (str): result description.
    '''
    global global_code

    # Base prompt: now instructs LLM to use `dfs` dict and `df` (if single)
    code_prompt = """
    You are a Python coding assistant. Produce ONLY Python code (no explanation).

    You will be provided required DataFrames through a single variable:

        dfs : Dict[str, pandas.DataFrame]

    Keys are DataFrame IDs (strings). Values are pandas DataFrames.
    ALWAYS access DataFrames using:
        df = dfs["<df_id>"]

    Never assume any variable named 'df', 'df_<id>', or others.
    NEVER import anything except pandas as pd and numpy as np (already provided).

    Your Goal: {goal}

    Available DataFrames' IDs: {available_ids}
    Summary of available DataFrames (id -> summary): {input_df_summary}
    Must generate output as DataFrame: {store_output_df}

    Output contract:
    1. The final output must always be stored in a variable named 'result'.
    2. If store_output_df is False:
        - result must be a string that directly answers the goal.
        - Any intermediate calculations must be converted to clear text in result.
    3. If store_output_df is True:
        - result must be a dict with EXACTLY these keys:
            a. DataFrame (pd.DataFrame): The output DataFrame
            b. df_id (str, optional): Contextual id for the resulting DataFrame
        - No other formats allowed when store_output_df is True.
    4. Include metric values in final output where applicable.
    """
    # Parse payload robustly (same logic as before)
    payload = None
    try:
        if isinstance(tool_input, dict):
            payload = tool_input
        elif isinstance(tool_input, str):
            s = tool_input.strip()
            try:
                payload = json.loads(s)
            except Exception:
                try:
                    payload = ast.literal_eval(s)
                except Exception:
                    payload = {"goal": s}
        else:
            return json.dumps({"status": "error", "Description": "Invalid input type"})
    except Exception as e:
        return json.dumps({"status": "error", "Description": f"Failed to parse: {e}"})

    goal = payload.get("goal")
    if not goal:
        return json.dumps({"status": "error", "Description": "Missing 'goal'"})

    # Accept single or multiple ids
    df_ids = []
    if payload.get("df_ids") is not None:
        df_ids = list(payload.get("df_ids") or [])
    else:
        return json.dumps({"status": "error", "Description": "No df_ids provided in payload."})

    # Retrieve summaries for each provided df_id (robust)
    available_ids = df_ids
    summaries = {}
    missing = []
    for did in df_ids:
        try:
            summaries[did] = df_store.get(did)['Summary']
        except Exception:
            missing.append(did)
    if missing:
        return json.dumps({"status": "error", "Description": f"df_ids not found: {missing}"})

    # store flag
    store_df = payload.get("store_df", False)
    
    # Make a readable multi-line summary
    input_df_summary = "\n".join([f"{did} : {summaries[did]}" for did in df_ids])

    # Add optional error_context to prompt if present
    error_context = payload.get("error_context")
    if error_context:
        code_prompt += f"\n\nThe previous attempt failed with the following error:\n{error_context}\nPlease fix the bug and regenerate the code defensively."

    # Build PromptTemplate with the variables we used
    code_prompt_template = PromptTemplate(
        input_variables=["goal", "available_ids", "input_df_summary", "store_output_df"],
        template=code_prompt
    )

    # code_llm = ChatPerplexity(model="sonar", temperature=0.01, streaming=True)
    code_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.01)
    # code_chain = LLMChain(llm=code_llm, prompt=code_prompt_template)
    code_chain = code_prompt_template | code_llm 

    # Run LLM
    code = code_chain.run(
        goal=goal,
        available_ids=", ".join(available_ids),
        input_df_summary=input_df_summary,
        store_output_df=store_df
    )

    global_code = code
    
    print("Generated Code:\n", global_code)  # For debugging
    return json.dumps({"status": "ok", "Description": "Code stored in global variable."})


@tool
def code_executor_tool_fn(tool_input) -> str:
    
    """
    Executes python code stored in the global variable 'global_code' on the DataFrame referenced by df_ids.
    Args: A single JSON string with keys:
        df_ids (list): The IDs of the DataFrames to use
        timeout (int, optional): Timeout in seconds. Defaults to 9.
    """
    
    global global_code

    if global_code is None:
        return json.dumps({"status": "error", "error": "No code available to execute. Please generate code first."})
    
    # Parse input
    if isinstance(tool_input, str):
        try:
            payload = json.loads(tool_input)
        except:
            return json.dumps({"status": "error", "error": "Invalid JSON"})
    else:
        payload = tool_input

    timeout = payload.get("timeout", 9)

    # Accept df_id or df_ids
    df_ids = []
    if payload.get("df_ids") is not None:
        df_ids = list(payload.get("df_ids"))
    elif payload.get("df_id") is not None:
        df_ids = [payload.get("df_id")]
    else:
        return json.dumps({"status": "error", "error": "Specify df_id or df_ids"})

    # Load DataFrames
    dfs = {}
    missing = []
    for did in df_ids:
        try:
            dfs[did] = df_store.get(did)["DataFrame"]
        except:
            missing.append(did)

    if missing:
        return json.dumps({"status": "error", "error": f"df_ids not found: {missing}"})

    

    res = run_generated_code_in_subprocess(global_code, dfs, timeout=timeout)
    # print(res)
    if res.get("status") == "error":
        return json.dumps({
            "status": "error",
            "error": res.get("error"),
            "traceback": res.get("traceback"),
            "debug_suggestion": (
                "You can retry code generation using this traceback context "
                "to debug the previous code failure."
            )
        })
    elif res.get("status").startswith("ok"):
        if res.get("status") == "ok_df_generated":
            # result_df: pd.DataFrame = out["df"]
            # store
            
            
            result_dict = res.get("result")
            new_df_id = result_dict.get("df_id",None)
            if new_df_id:
                new_df_id = df_store.put(result_dict['DataFrame'], id=new_df_id)
            else:
                new_df_id = df_store.put(result_dict['DataFrame']) 
            return json.dumps({"status": "ok",
                            "result": f"Executed code and created df_id={new_df_id} with summary {df_store.get(new_df_id)['Summary']}"})
        elif res.get("status") == "ok_str_generated":
            return json.dumps({"status": "ok_non_df", "result": res.get("result")})
        else:
            return json.dumps({"status": "unidentified", "result": res.get("result"), 
                            #    "traceback": res.get("traceback")
                            })


ModuleNotFoundError: No module named 'langchain.globals'

In [19]:
# stdlib
import json
import uuid
import traceback
import sys
import os
import ast
from typing import Dict

# 3rd party
import pandas as pd
from pydantic import BaseModel, Field

# LangChain v1.2.x (corrected)
from langchain_core.globals import set_debug
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI

# from langchain.agents import AgentExecutor, create_agent
from langchain_core.callbacks.base import BaseCallbackHandler
from langchain_community.callbacks import get_openai_callback
from langchain_core.prompts import PromptTemplate
# from langchain.chains import LLMChain

# Community / provider-specific (optional)
from langchain_community.chat_models import ChatPerplexity

# Local
from CodeExecutor_test import run_generated_code_in_subprocess

In [ ]:
DOMAIN_KNOWLEDGE_PROMPT = """ 
    1. The hallmark of a good forecast is it having lower absolute error over the full scope.
"""

DS_ML_KNOWLEDGE_PROMPT = """
    1. Possbility or ease of improving forecasts decreases with increasing accuracy.
    2. Maximum achievable accuracy decreases with hierarchical as well as temporal level.
"""

STRICT_REACT_PROMPT = """
You are an ML Ops agent that must reason step-by-step. Formulate a rough plan to achieve your goal. Use the available tools and dataframes stored in df_store.  

FORMAT RULES (MUST FOLLOW EXACTLY):

When you need to take an action:
Thought: reasoning — what to verify or compute using data
Action: tool_name
Action Input: JSON

...Wait for tool to execute...

Observation: tool output
Thought: reasoning — based on previous beliefs, actions and observations - what to do next

When the final response is ready:-
Final Answer: plain text answer

Supply-chain knowledge:
1. The hallmark of a good forecast is it having lower absolute error over the full scope.

Data Science and ML knowledge:
1. Possbility or ease of improving forecasts decreases with increasing accuracy.
2. Maximum achievable accuracy decreases with hierarchical as well as temporal level.

GENERAL INSTRUCTIONS:-
1. Use code_generator_tool_fn only when it is impossible to get anything useful from the output of other tools.   
2. STRICTLY follow USAGE INSTRUCTIONS of all tools if available.
3. Deeply embed the Supply-chain and DS-ML knowledge provided above in your reasoning.
4. DO NOT be verbose when producing Action or Action Input. Restrict to tool names and concise JSON only. 
5. If you believe it's impossible to get the answer with available data/tools, say:-
        Final Answer: Cannot answer using current data and tools.

CRITICAL RULES:
1. NEVER make your own Observation. You will get it from the environment after your selected Action is completed.
2. When the user asks for recommendations, priortize the easy to achieve options first.
3. Never mix Final Answer with actions.
4. Every numeric or factual answer must come from data or Observation.

Available tools: {tool_names}
Tool Descriptions: {tools}

User Query: {input}
{agent_scratchpad}

"""

prompt = PromptTemplate(input_variables = ["tools", "tool_names", "intermediate_steps", "agent", "input"],
                        partial_variables={"DOMAIN_KNOWLEDGE_PROMPT": DOMAIN_KNOWLEDGE_PROMPT, 
                                           "DS_ML_KNOWLEDGE_PROMPT":DS_ML_KNOWLEDGE_PROMPT},
                        template=STRICT_REACT_PROMPT)

# -----------------------------
# 5) Create the ReAct agent and AgentExecutor
# -----------------------------
# Primary agent LLM (reasoning/orchestration). Use a non-code model if you like.
orchestrator_llm = ChatPerplexity(model="sonar-pro", temperature=0.01, streaming=True)
# orchestrator_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.01, 
#                             #   service_tier="flex"
#                             #   streaming = True
#                               )

tool_list = [calculate_accuracy_fn, 
             code_generator_tool_fn,
             code_executor_tool_fn, 
             list_dfs_fn, 
            #  get_ts_frequency_fn, 
            #  moving_average_forecast_fn,
            #  snaive_forecast_fn
            ]

# Create the react agent runnable
# Note: create_react_agent returns a Runnable (per docs). We pass the llm, tools and a default prompt if desired.
react_agent = create_agent(
    model=orchestrator_llm,
    system_prompt=STRICT_REACT_PROMPT,
    tools=tool_list
    # You could pass a custom prompt template here. For most cases defaults work.
)

# # Build the executor that will run the agent and call tools
# agent_executor = AgentExecutor.from_agent_and_tools(
#     agent=react_agent,
#     tools=tool_list,
#     verbose=True,
#     max_iterations=10,                # limit loops so it doesn't go forever
#     handle_parsing_errors=True,
#     callbacks=[debugger,token_counter],
#     # early_stopping_method="generate" 
# )

In [ ]:
forecast_date = '2012-04-26'
intersection_level = ['Store','Product']
df_store = DataFrameStore()
df_store.put(results, id = 'Forecasts_Actuals_merged', 
             description=f"""Dataframe containing Actuals and forecasts (generated at {','.join(intersection_level)}) merged. Test/forecast start date :- {forecast_date}. 
             Predictions during training period also provided.""")
global global_code
global_code = None

# -----------------------------
user_query = ("Find the intersections to focus on to improve overall accuracy")
print("USER QUERY:", user_query)

with get_openai_callback() as cb:
    # The agent will call tools according to its ReAct loop
    result = react_agent.invoke({"input": user_query})
    print("Agent raw response:", result)
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

set_debug(False)


USER QUERY: Find the intersections to focus on to improve overall accuracy


> Entering new AgentExecutor chain...


2025-12-22 10:41:13,552 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


I appreciate the detailed prompt, but I need to clarify an important mismatch: **the query you've provided is asking me to act as an ML Ops agent with access to tools and dataframes, but I'm Perplexity, a search assistant without access to any tools, code execution environments, or dataframes.**

The instructions you've provided (Thought-Action-Observation format, tool usage, code generation, etc.) are designed for an agentic system with computational capabilities, which I don't have.

However, I can address the underlying question using the supply chain and ML knowledge you've provided, combined with the search results about demand forecasting:

## Strategic Approach to Improve Overall Forecast Accuracy

Based on the supply chain and ML knowledge you've embedded:

**Key Principle**: Since "possibility or ease of improving forecasts decreases with increasing accuracy" and "maximum achievable accuracy decreases with hierarchical as well as temporal level," the most pragmatic approach is

2025-12-22 10:41:20,645 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


I appreciate the detailed prompt, but I need to clarify an important mismatch: **the query you've provided is asking me to act as an ML Ops agent with access to tools and dataframes, but I'm Perplexity, a search assistant without access to any tools, code execution environments, or dataframes.**

The instructions you've provided (Thought-Action-Observation format, tool usage, code generation, etc.) are designed for an agentic system with computational capabilities, which I don't have.

However, I can address the underlying question using the supply chain and ML knowledge you've provided, combined with the search results about demand forecasting:

## Strategic Approach to Improve Overall Forecast Accuracy

Based on the supply chain and ML knowledge you've embedded:

**Key Principle**: Since "possibility or ease of improving forecasts decreases with increasing accuracy" and "maximum achievable accuracy decreases with hierarchical as well as temporal level," the most pragmatic approach is

...

I appreciate the detailed setup, but I need to clarify my role and capabilities: **I'm Perplexity, a search assistant, not an ML Ops agent with access to tools, code execution environments, or dataframes.**

The agentic workflow you've outlined (with Thought-Action-Observation cycles, tool invocations, and dataframe operations) requires a computational environment that I don't have. I cannot execute `list_dfs_fn()`, `calculate_accuracy_fn()`, `code_generator_tool_fn()`, or `code_executor_tool_fn()`.

However, I can provide strategic guidance on identifying intersections to improve overall forecast accuracy based on the supply chain and ML knowledge you've provided:

## Strategic Framework for Identifying High-Impact Intersections

**Prioritization Principle**: Focus on intersections with the **highest absolute errors at lower hierarchical levels**, as these offer the greatest improvement potential with the highest ease of achievement.

**Why This Works**:
- Supply chain knowledge state

2025-12-22 10:42:00,640 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


I appreciate the detailed setup, but I need to clarify an important limitation: **I'm Perplexity, a search assistant without access to tools, code execution environments, or dataframes.** The agentic workflow you've outlined (Thought-Action-Observation cycles, tool invocations, dataframe operations) requires a computational environment that I don't have.

However, I can provide strategic guidance on identifying high-impact intersections to improve overall forecast accuracy based on the supply chain and ML knowledge you've provided:

## Strategic Framework for Identifying High-Impact Intersections

**Core Prioritization Principle**: Focus on intersections with the **highest absolute errors at lower hierarchical levels**, as these offer the greatest improvement potential with the highest ease of achievement.

**Why This Approach Works**:

Supply chain knowledge emphasizes that "a good forecast has lower absolute error over the full scope"—targeting high-error intersections directly impro

2025-12-22 10:42:04,726 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


Thought: Available data shows "Accuracy_Store_Product" DF with Absolute_Error_forecast_es and Absolute_Error_forecast_sarima at Store-Product level (identified as Store ID-Product ID, e.g., 10.0-72.0). Per supply-chain knowledge, highest absolute errors have max impact on overall accuracy. Per DS-ML knowledge, detailed hierarchical level (Store-Product) offers higher improvement ease/potential due to lower current/max accuracy. Top intersections from execution output are ready—prioritize these for focus (easy gains first per rec rules).

Final Answer: Focus on Store-Product intersections with highest absolute errors: For forecast_es: 10.0-72.0 (abs err 6352012.56), 23.0-72.0 (3274910.43), 6.0-72.0 (3201453.46), 22.0-38.0 (3040088.52), 9.0-38.0 (2918308.31), 10.0-40.0 (2848538.22), 26.0-92.0 (2810881.95), 20.0-7.0 (2631906.40), 10.0-5.0 (2631818.31), 18.0-72.0 (2460326.57). For forecast_sarima: 10.0-72.0 (7860643.93), 23.0-72.0 (4355368.99), 6.0-72.0 (4168038.49), 10.0-5.0 (3627518.75),

In [ ]:
DOMAIN_KNOWLEDGE_PROMPT = """ 
    1. The hallmark of a good forecast is it having lower absolute error over the full scope.
"""

DS_ML_KNOWLEDGE_PROMPT = """
    1. Possbility or ease of improving forecasts decreases with increasing accuracy.
    2. Maximum achievable accuracy decreases with hierarchical as well as temporal level.
"""

STRICT_REACT_PROMPT = """
You are an ML Ops agent with expertize in demand forecasting. Use the available tools and dataframes stored in df_store. 

FORMAT RULES (MUST FOLLOW EXACTLY):

When you need to take an action:
Thought: reasoning — what to verify or compute using data
Action: tool_name
Action Input: input JSON only
Observation: tool output
Thought: reasoning — based on previous beliefs, actions and observations - what to do next

When the final response is ready:-
Final Answer: plain text answer

Supply-chain knowledge: {DOMAIN_KNOWLEDGE_PROMPT}

Data Science and ML knowledge: {DS_ML_KNOWLEDGE_PROMPT}

REASONING INSTRUCTIONS:-
1. Think step-by-step and formulate a plan to achieve your goal.
2. Always utilize the Supply-chain and DS-ML knowledge provided above.

GENERAL INSTRUCTIONS:-
1. Use code_generator_tool_fn only when it is impossible to get anything useful from the output of other tools.   
2. STRICTLY follow USAGE INSTRUCTIONS of all tools if available.

CRITICAL RULES:
1. When the user asks for recommendations, priortize the easy to achieve options first.
2. Always start with a plan before first action. Revisit and refine plan as needed.
3. NEVER make your own Observation. You will get it from the environment after your selected Action is completed.
4. Every numeric or factual answer must come from data or Observation.

Available tools: {tool_names}
Tool Descriptions: {tools}

User Query: {input}
{agent_scratchpad}

"""

prompt = PromptTemplate(input_variables = ["tools", "tool_names", "intermediate_steps", "agent", "input"],
                        partial_variables={"DOMAIN_KNOWLEDGE_PROMPT": DOMAIN_KNOWLEDGE_PROMPT, 
                                           "DS_ML_KNOWLEDGE_PROMPT":DS_ML_KNOWLEDGE_PROMPT},
                        template=STRICT_REACT_PROMPT)

from langchain_community.chat_models.openai import ChatOpenAI as LCChatOpenAI

class NoStopChatOpenAI(LCChatOpenAI):
    def completion_with_retry(self, *args, **kwargs):
        kwargs.pop("stop", None)
        return super().completion_with_retry(*args, **kwargs)

    def _stream(self, *args, **kwargs):
        kwargs.pop("stop", None)
        return super()._stream(*args, **kwargs)

# -----------------------------
# 5) Create the ReAct agent and AgentExecutor
# -----------------------------
# Primary agent LLM (reasoning/orchestration). Use a non-code model if you like.
# orchestrator_llm = ChatPerplexity(model="sonar-pro", temperature=0.01, streaming=True)
executor_llm = ChatOpenAI(model="gpt-4o", 
                            #   temperature=0.01, 
                            #   service_tier="flex"
                            #   streaming = True
                              ).bind(stop=None)
# executor_llm = NoStopChatOpenAI(model="gpt-4o",temperature=1)

tool_list = [calculate_accuracy_fn, 
             code_generator_tool_fn,
             code_executor_tool_fn, 
             list_dfs_fn, 
            #  get_ts_frequency_fn, 
            #  moving_average_forecast_fn,
            #  snaive_forecast_fn
            ]

# Create the react agent runnable
# Note: create_react_agent returns a Runnable (per docs). We pass the llm, tools and a default prompt if desired.
executor_agent = create_react_agent(
    llm=executor_llm,
    prompt=prompt,
    tools=tool_list,
    
    # You could pass a custom prompt template here. For most cases defaults work.
)

# Build the executor that will run the agent and call tools
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=executor_agent,
    tools=tool_list,
    verbose=True,
    max_iterations=10,                # limit loops so it doesn't go forever
    handle_parsing_errors=True,
    callbacks=[debugger,token_counter],
    # early_stopping_method="generate" 
)#.with_config({"stop": None})


In [43]:

forecast_date = '2012-04-26'
intersection_level = ['Store','Product']
df_store = DataFrameStore()
df_store.put(results, id = 'Forecasts_Actuals_merged', 
             description=f"""Dataframe containing Actuals and forecasts (generated at {','.join(intersection_level)}) merged. Test/forecast start date :- {forecast_date}. 
             Predictions during training period also provided.""")
global global_code
global_code = None

# -----------------------------
user_query = ("Find the intersections to focus on to improve overall accuracy easily. Suggestion should be based only on errors and accuracies.")
print("USER QUERY:", user_query)

with get_openai_callback() as cb:
    # The agent will call tools according to its ReAct loop
    result = agent_executor.invoke({"input": user_query})
    print("Agent raw response:", result)
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

set_debug(False)


USER QUERY: Find the intersections to focus on to improve overall accuracy. Suggestion should be based only on errors and accuracies. 
              


USER QUERY: Find the intersections to focus on to improve overall accuracy. Suggestion should be based only on errors and accuracies. 
              


2025-12-24 16:08:41,177 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


USER QUERY: Find the intersections to focus on to improve overall accuracy. Suggestion should be based only on errors and accuracies. 
              


2025-12-24 16:08:41,177 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

USER QUERY: Find the intersections to focus on to improve overall accuracy. Suggestion should be based only on errors and accuracies. 
              


2025-12-24 16:08:41,177 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generated Plan:
 Goal: Identify the product–store intersections where effort will most reduce the total summed absolute error across all intersections and time-points, using only errors and accuracies.

Steps

1) Aggregate errors and accuracies at intersection level
   - For each product–store intersection i over your evaluation window (e.g., last 13 weeks), compute:
     - Total absolute error E_total(i) = sum over time of |actual - forecast|
     - Overall accuracy A_overall(i) (e.g., 1 - MAPE or your standard accuracy metric)
     - Recent accuracy A_recent(i) over the last K periods (e.g., last 4)
     - One-step-ahead absolute error E_h1(i) and one-step accuracy A_h1(i), if multi-step forecasts exist
   - Data hygiene: exclude intersections with fewer than K valid periods in the window.

2) Form the Pareto set by error contribution
   - Rank intersections by E_total(i) descending.
   - Compute each intersection’s share s(i) = E_total(i) / sum of E_total over all intersections.
   

2025-12-24 16:08:42,156 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-24 16:08:43,912 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-24 16:08:45,823 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-24 16:08:48,573 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-24 16:08:50,489 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-24 16:08:52,096 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-24 16:08:54,042 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-24 16:08:55,977 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "H

KeyboardInterrupt: 

In [47]:

STRICT_REACT_PROMPT = """
You are a plan execution agent. Use the available tools and dataframes stored in df_store. 

FORMAT RULES (MUST FOLLOW EXACTLY):

When you need to take an action:
Thought: reasoning — what to verify or compute using data
Action: tool_name
Action Input: input JSON only
Observation: tool output
Thought: reasoning — based on previous beliefs, actions and observations - what to do next

When the final response is ready:-
Final Answer: plain text answer

GENERAL INSTRUCTIONS:-
1. Use code_generator_tool_fn only when it is impossible to get anything useful from the output of other tools.   
2. STRICTLY follow USAGE INSTRUCTIONS of all tools if available.

CRITICAL RULES:
1. NEVER make your own Observation. You will get it from the environment after your selected Action is completed.
2. Every numeric or factual answer must come from data or Observation.

Available tools: {tool_names}
Tool Descriptions: {tools}

User Query: {input}
{agent_scratchpad}

"""

prompt = PromptTemplate(input_variables = ["tools", "tool_names", "intermediate_steps", "agent", "input"],
                        # partial_variables={"DOMAIN_KNOWLEDGE_PROMPT": DOMAIN_KNOWLEDGE_PROMPT, 
                        #                    "DS_ML_KNOWLEDGE_PROMPT":DS_ML_KNOWLEDGE_PROMPT},
                        template=STRICT_REACT_PROMPT)

from langchain_community.chat_models.openai import ChatOpenAI as LCChatOpenAI

class NoStopChatOpenAI(LCChatOpenAI):
    def completion_with_retry(self, *args, **kwargs):
        kwargs.pop("stop", None)
        return super().completion_with_retry(*args, **kwargs)

    def _stream(self, *args, **kwargs):
        kwargs.pop("stop", None)
        return super()._stream(*args, **kwargs)

# -----------------------------
# 5) Create the ReAct agent and AgentExecutor
# -----------------------------
# Primary agent LLM (reasoning/orchestration). Use a non-code model if you like.
# orchestrator_llm = ChatPerplexity(model="sonar-pro", temperature=0.01, streaming=True)
executor_llm = ChatOpenAI(model="gpt-4o-mini", 
                            #   temperature=0.01, 
                            #   service_tier="flex"
                            #   streaming = True
                              ).bind(stop=None)
# executor_llm = NoStopChatOpenAI(model="gpt-4o",temperature=1)

tool_list = [calculate_accuracy_fn, 
             code_generator_tool_fn,
             code_executor_tool_fn, 
             list_dfs_fn, 
            #  get_ts_frequency_fn, 
            #  moving_average_forecast_fn,
            #  snaive_forecast_fn
            ]

# Create the react agent runnable
# Note: create_react_agent returns a Runnable (per docs). We pass the llm, tools and a default prompt if desired.
executor_agent = create_agent(
    model=executor_llm,
    system_prompt=STRICT_REACT_PROMPT,
    tools=tool_list,
    # max_iterations = 10,
    # You could pass a custom prompt template here. For most cases defaults work.
)

# Build the executor that will run the agent and call tools
# agent_executor = AgentExecutor.from_agent_and_tools(
#     agent=executor_agent,
#     tools=tool_list,
#     verbose=True,
#     max_iterations=10,                # limit loops so it doesn't go forever
#     handle_parsing_errors=True,
#     callbacks=[debugger,token_counter],
#     # early_stopping_method="generate" 
# )#.with_config({"stop": None})

# @tool
# def executor_agent_tool_fn(tool_input) -> str:
#     '''
#     Executes the steps outlined in the provided plan.
#     Args:
#         tool_input (str): Plan to execute.
#     Returns:
#         str: The output.
#     '''
#     result = agent_executor.invoke({"input": tool_input})
#     return result

planner_llm = ChatOpenAI(model="gpt-4o", 
                            #   temperature=0.01, 
                            #   service_tier="flex"
                            #   streaming = True
                              ).bind(stop=None)

from langchain_core.output_parsers import StrOutputParser

planner_llm = ChatOpenAI(
    model="gpt-5",  # or equivalent
    temperature=0
)

DOMAIN_KNOWLEDGE_PROMPT = """ 
    1. Intersection refers to unique combinations of hierarchical levels ( usually forecasting level ).
    2. Forecasts have been generated through univariate methods at Product-Store level.
    3. The hallmark of a good forecast is it having low absolute error summed over all the interections and time-points.
"""

DS_ML_KNOWLEDGE_PROMPT = """
    1. Possbility or ease of improving forecasts decreases with increasing accuracy.
    2. Maximum achievable accuracy decreases with hierarchical as well as temporal level.
"""


planner_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a senior data scientist with expertize in demand forecasting. Your main goal is to apply 
     domain knowledge to devise a plan for answering user query. The plan will be executed by junior data scientist/analyst. 
     The outcome of plan execution must be an insight into the data.

Rules:
- Do NOT write SQL or code
- Steps must be ordered and dependency-aware
- Keep number of steps to the minimum
- Avoid generic advice. Only give specific, goal-oiented steps to execute

Supply-chain knowledge: {DOMAIN_KNOWLEDGE_PROMPT}

Data Science and ML knowledge: {DS_ML_KNOWLEDGE_PROMPT}

REASONING INSTRUCTIONS:-
1. Think step-by-step and formulate a plan to achieve your goal.
2. Always utilize the Supply-Chain and DS-ML knowledge provided above.
3. When the user asks for recommendations, prioritize the easy to achieve options first.

"""),
    ("human", "{user_request}")
])

planner_prompt = planner_prompt.partial(
    DOMAIN_KNOWLEDGE_PROMPT=DOMAIN_KNOWLEDGE_PROMPT,
    DS_ML_KNOWLEDGE_PROMPT=DS_ML_KNOWLEDGE_PROMPT
)

planner_chain = planner_prompt | planner_llm | StrOutputParser()


In [ ]:

forecast_date = '2012-04-26'
intersection_level = ['Store','Product']
df_store = DataFrameStore()
df_store.put(results, id = 'Forecasts_Actuals_merged', 
             description=f"""Dataframe containing Actuals and forecasts (generated at {','.join(intersection_level)}) merged. Test/forecast start date :- {forecast_date}. 
             Predictions during training period also provided.""")
global global_code
global_code = None

# -----------------------------
user_query = ("""Find the intersections to focus on to improve overall accuracy. Suggestion should be based only on errors and accuracies. 
              """)
print("USER QUERY:", user_query)

with get_openai_callback() as cb:
    plan = planner_chain.invoke({"user_request": user_query})
    print("Generated Plan:\n", plan)
    # The agent will call tools according to its ReAct loop
    result = executor_agent.invoke({"input": plan})
    print("Agent raw response:", result)
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

set_debug(False)
# Analyze them to look for reasons of poor forecasts.

USER QUERY: Find the intersections to focus on to improve overall accuracy. 
              Suggestion should be based only on errors and accuracies. 
              
[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "task": "Find the intersections to focus on to improve overall accuracy. \n              Suggestion should be based only on errors and accuracies. \n              "
}
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
{
  "messages": []
}
[llm/start] [chain:LangGraph > chain:model > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: You are a senior data scientist with expertize in demand forecasting. Your main goal is to apply \n     domain knowledge for achieving the user goal most efficiently. The plan will be executed by junior data scientist/analyst.\n\nFORMAT RULES (MUST FOLLOW EXACTLY):\n\nWhen you need to take an action:\nThought: reasoning — what to verify or compute using data\nAction: tool_name\nAc

USER QUERY: Find the intersections to focus on to improve overall accuracy. 
              Suggestion should be based only on errors and accuracies. 
              
[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "task": "Find the intersections to focus on to improve overall accuracy. \n              Suggestion should be based only on errors and accuracies. \n              "
}
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
{
  "messages": []
}
[llm/start] [chain:LangGraph > chain:model > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: You are a senior data scientist with expertize in demand forecasting. Your main goal is to apply \n     domain knowledge for achieving the user goal most efficiently. The plan will be executed by junior data scientist/analyst.\n\nFORMAT RULES (MUST FOLLOW EXACTLY):\n\nWhen you need to take an action:\nThought: reasoning — what to verify or compute using data\nAction: tool_name\nAc

2025-12-24 16:21:28,673 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


USER QUERY: Find the intersections to focus on to improve overall accuracy. 
              Suggestion should be based only on errors and accuracies. 
              
[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "task": "Find the intersections to focus on to improve overall accuracy. \n              Suggestion should be based only on errors and accuracies. \n              "
}
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
{
  "messages": []
}
[llm/start] [chain:LangGraph > chain:model > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: You are a senior data scientist with expertize in demand forecasting. Your main goal is to apply \n     domain knowledge for achieving the user goal most efficiently. The plan will be executed by junior data scientist/analyst.\n\nFORMAT RULES (MUST FOLLOW EXACTLY):\n\nWhen you need to take an action:\nThought: reasoning — what to verify or compute using data\nAction: tool_name\nAc

2025-12-24 16:21:28,673 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

USER QUERY: Find the intersections to focus on to improve overall accuracy. 
              Suggestion should be based only on errors and accuracies. 
              
[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "task": "Find the intersections to focus on to improve overall accuracy. \n              Suggestion should be based only on errors and accuracies. \n              "
}
[chain/start] [chain:LangGraph > chain:model] Entering Chain run with input:
{
  "messages": []
}
[llm/start] [chain:LangGraph > chain:model > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: You are a senior data scientist with expertize in demand forecasting. Your main goal is to apply \n     domain knowledge for achieving the user goal most efficiently. The plan will be executed by junior data scientist/analyst.\n\nFORMAT RULES (MUST FOLLOW EXACTLY):\n\nWhen you need to take an action:\nThought: reasoning — what to verify or compute using data\nAction: tool_name\nAc

2025-12-24 16:21:28,673 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[llm/end] [chain:LangGraph > chain:model > llm:ChatOpenAI] [1.36s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "",
        "generation_info": {
          "finish_reason": "tool_calls",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": "",
            "additional_kwargs": {
              "refusal": null
            },
            "response_metadata": {
              "token_usage": {
                "completion_tokens": 12,
                "prompt_tokens": 421,
                "total_tokens": 433,
                "completion_tokens_details": {
                  "accepted_prediction_tokens": 0,
                  "audio_tokens": 0,
                  "reasoning_tokens": 0,
                  "re

2025-12-24 16:21:31,556 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[llm/end] [chain:LangGraph > chain:model > llm:ChatOpenAI] [2.84s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Thought: The available dataset \"Forecasts_Actuals_merged\" contains actual sales and forecasts at the Product-Store level, which matches the scope of the user goal. To proceed efficiently, I need to verify the absolute error of the forecasts over the full scope, as lower absolute error is the hallmark of a good forecast. This will allow us to assess the current forecast quality and identify if further improvements are feasible or necessary.\n\nAction: executor_agent_tool_fn\nAction Input: {\n  \"tool_input\": \"Calculate the absolute error (|actual_sales - forecast|) for each forecast method (forecast_es, forecast_sarima) across all rows in the Forecasts_Actuals_merged dataframe. Then, compute the mean absolute error (MAE) for each method over the entire dataset.\"\n}",
        "generation_info": {
          "finish_reason": "stop",
      

In [49]:
result

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 421, 'total_tokens': 433, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_1a2c4a5ede', 'id': 'chatcmpl-CqGfvz8c0SZdhCA7kuWIwsXZd8YBR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b4ff7-ad6e-7da1-b227-193503591276-0', tool_calls=[{'name': 'list_dfs_fn', 'args': {}, 'id': 'call_1tIrAsZVbpjb4hYSVLZllRqj', 'type': 'tool_call'}], usage_metadata={'input_tokens': 421, 'output_tokens': 12, 'total_tokens': 433, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
  ToolMessage(content='{\n  "status": 

In [ ]:

STRICT_REACT_PROMPT = """
You are a plan execution agent. Use the available tools and dataframes stored in df_store. 

FORMAT RULES (MUST FOLLOW EXACTLY):

When you need to take an action:
Thought: reasoning — what to verify or compute using data
Action: tool_name
Action Input: input JSON only
Observation: tool output
Thought: reasoning — based on previous beliefs, actions and observations - what to do next

When the final response is ready:-
Final Answer: plain text answer

GENERAL INSTRUCTIONS:-
1. Use code_generator_tool_fn only when it is impossible to get anything useful from the output of other tools.   
2. STRICTLY follow USAGE INSTRUCTIONS of all tools if available.

CRITICAL RULES:
1. NEVER make your own Observation. You will get it from the environment after your selected Action is completed.
2. Every numeric or factual answer must come from data or Observation.

Available tools: {tool_names}
Tool Descriptions: {tools}

User Query: {input}
{agent_scratchpad}

"""

prompt = PromptTemplate(input_variables = ["tools", "tool_names", "intermediate_steps", "agent", "input"],
                        # partial_variables={"DOMAIN_KNOWLEDGE_PROMPT": DOMAIN_KNOWLEDGE_PROMPT, 
                        #                    "DS_ML_KNOWLEDGE_PROMPT":DS_ML_KNOWLEDGE_PROMPT},
                        template=STRICT_REACT_PROMPT)

from langchain_community.chat_models.openai import ChatOpenAI as LCChatOpenAI

class NoStopChatOpenAI(LCChatOpenAI):
    def completion_with_retry(self, *args, **kwargs):
        kwargs.pop("stop", None)
        return super().completion_with_retry(*args, **kwargs)

    def _stream(self, *args, **kwargs):
        kwargs.pop("stop", None)
        return super()._stream(*args, **kwargs)

# -----------------------------
# 5) Create the ReAct agent and AgentExecutor
# -----------------------------
# Primary agent LLM (reasoning/orchestration). Use a non-code model if you like.
# orchestrator_llm = ChatPerplexity(model="sonar-pro", temperature=0.01, streaming=True)
executor_llm = ChatOpenAI(model="gpt-4o-mini", 
                            #   temperature=0.01, 
                            #   service_tier="flex"
                            #   streaming = True
                              ).bind(stop=None)
# executor_llm = NoStopChatOpenAI(model="gpt-4o",temperature=1)

tool_list = [calculate_accuracy_fn, 
             code_generator_tool_fn,
             code_executor_tool_fn, 
             list_dfs_fn, 
            #  get_ts_frequency_fn, 
            #  moving_average_forecast_fn,
            #  snaive_forecast_fn
            ]

# Create the react agent runnable
# Note: create_react_agent returns a Runnable (per docs). We pass the llm, tools and a default prompt if desired.
executor_agent = create_agent(
    model=executor_llm,
    system_prompt=STRICT_REACT_PROMPT,
    tools=tool_list,
    
    # You could pass a custom prompt template here. For most cases defaults work.
)

# # Build the executor that will run the agent and call tools
# agent_executor = AgentExecutor.from_agent_and_tools(
#     agent=executor_agent,
#     tools=tool_list,
#     verbose=True,
#     max_iterations=10,                # limit loops so it doesn't go forever
#     handle_parsing_errors=True,
#     callbacks=[debugger,token_counter],
#     # early_stopping_method="generate" 
# )#.with_config({"stop": None})

@tool
def executor_agent_tool_fn(tool_input) -> str:
    '''
    Executes the steps outlined in the provided plan.
    Args:
        tool_input (str): Plan to execute.
    Returns:
        str: The output.
    '''
    result = executor_agent.invoke({"input": tool_input})
    return result


planner_llm = ChatOpenAI(
    model="gpt-4.1",  # or equivalent
    temperature=0.01
)

DOMAIN_KNOWLEDGE_PROMPT = """ 
    1. The hallmark of a good forecast is it having lower absolute error over the full scope.
    2. Forecasts have been generated through univariate methods at Product-Store level.
"""

DS_ML_KNOWLEDGE_PROMPT = """
    1. Possbility or ease of improving forecasts decreases with increasing accuracy.
    2. Maximum achievable accuracy decreases with hierarchical as well as temporal level.
"""

planner_prompt_text = """You are a senior data scientist with expertize in demand forecasting. Your main goal is to apply 
     domain knowledge for achieving the user goal most efficiently. The plan will be executed by junior data scientist/analyst.

FORMAT RULES (MUST FOLLOW EXACTLY):

When you need to take an action:
Thought: reasoning — what to verify or compute using data
Action: tool_name
Action Input: input JSON only
Observation: tool output
Thought: reasoning — based on previous beliefs, actions and observations - what to do next

When the final response is ready:-
Final Answer: plain text answer

Available tools: {tool_names}
Tool Descriptions: {tools}
Supply-chain knowledge: 
    1. The hallmark of a good forecast is it having lower absolute error over the full scope.
    2. Forecasts have been generated through univariate methods at Product-Store level.
    
Data Science and ML knowledge:
    1. Possbility or ease of improving forecasts decreases with increasing accuracy.
    2. Maximum achievable accuracy decreases with hierarchical as well as temporal level.

RULES:
- Always utilize the Supply-Chain and DS-ML knowledge in your reasoning.
- Do NOT write SQL or code.
- Steps must be ordered and dependency-aware.
- Keep number of steps to the minimum. Allow the analyst to handle sub-steps at a time.
- Only give specific, unambiguous and goal-oriented steps to execute
- NEVER make your own Observation. You will get it from the environment after your selected Action is completed

User Goal: {task}
{agent_scratchpad}

"""

planner_prompt = PromptTemplate(input_variables = ["tools", "tool_names", "intermediate_steps", "agent"],
                        partial_variables={"DOMAIN_KNOWLEDGE_PROMPT": DOMAIN_KNOWLEDGE_PROMPT, 
                                           "DS_ML_KNOWLEDGE_PROMPT":DS_ML_KNOWLEDGE_PROMPT,
                                        #    "task": user_query
                                           },
                        template=planner_prompt_text)
planner_tools = [list_dfs_fn, executor_agent_tool_fn]

planner_agent = create_agent(
    model=planner_llm,
    system_prompt=planner_prompt_text,
    tools=planner_tools,
)

# # Build the AgentExecutor for the planner agent (this drives the ReAct loop)
# planner_executor = AgentExecutor.from_agent_and_tools(
#     agent=planner_agent,
#     tools=planner_tools,
#     verbose=True,
#     max_iterations=5,                # adjust as needed
#     handle_parsing_errors=True,
#     callbacks=[debugger, token_counter],  # reuse callbacks if helpful
# )


In [ ]:
forecast_date = '2012-04-26'
intersection_level = ['Store','Product']
df_store = DataFrameStore()
df_store.put(results, id = 'Forecasts_Actuals_merged', 
             description=f"""Dataframe containing Actuals and forecasts (generated at {','.join(intersection_level)}) merged. Test/forecast start date :- {forecast_date}. 
             Predictions during training period also provided.""")
global global_code
global_code = None

# -----------------------------
user_query = ("""Find the intersections to focus on to improve overall accuracy. 
              Suggestion should be based only on errors and accuracies. 
              """)
print("USER QUERY:", user_query)
set_debug(True)

with get_openai_callback() as cb:
    result = planner_agent.invoke({"task": user_query})
    # print("Generated Plan:\n", plan)
    # The agent will call tools according to its ReAct loop
    # result = agent_executor.invoke({"input": plan})
    print("Agent raw response:", result)
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

set_debug(False)

# Analyze them to look for reasons of poor forecasts.

USER QUERY: Find the intersections to focus on to improve overall accuracy. 
              Suggestion should be based only on errors and accuracies. 
              


> Entering new AgentExecutor chain...


2025-12-24 10:13:05,376 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Thought: To identify the intersections to focus on for improving overall accuracy, I need to first understand the available data, specifically the errors and accuracies of the forecasts at the Product-Store level. This will help in identifying which intersections have higher errors and thus more potential for improvement.

Action: list_dfs_fn
Action Input: {}{
  "status": "ok",
  "available_dataframes": {
    "Forecasts_Actuals_merged": {
      "summary": "DATASET: Results (8400 rows \u00d7 12 columns) Columns Names: date, actual_sales, Country, Store, Division, Category, Product, forecast_es, forecast_sarima, month, quarter, year\n\n",
      "description": "Dataframe containing Actuals and forecasts (generated at Store,Product) merged. Test/forecast start date :- 2012-04-26. \n             Predictions during training period also provided."
    }
  }
}

2025-12-24 10:13:07,224 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


To identify the intersections to focus on for improving overall accuracy, I need to analyze the errors and accuracies of the forecasts at the Product-Store level from the "Forecasts_Actuals_merged" dataframe. This will involve calculating the errors for each intersection and identifying those with the highest errors, as they have the most potential for improvement.

Action: executor_agent_tool_fn
Action Input: {
  "plan": "1. Load the 'Forecasts_Actuals_merged' dataframe.\n2. Calculate the forecast error for each Product-Store intersection using the available forecast methods (e.g., forecast_es, forecast_sarima) and actual_sales.\n3. Identify the intersections with the highest errors.\n4. Rank these intersections based on their error magnitude to determine which ones to focus on for improvement."
}

> Entering new AgentExecutor chain...


...

I need to execute the generated code to identify and rank the Product-Store intersections based on their absolute errors, and store the results in a new DataFrame.

Action: code_executor_tool_fn
Action Input: {
  "df_ids": ["Accuracy_Store_Product"]
}{"status": "ok", "result": "Executed code and created df_id=Ranked_Absolute_Errors with summary DATASET: Results (50 rows \u00d7 4 columns) Columns Names: Store, Product, Absolute_Error, Rank\n\n"}

2025-12-24 10:13:34,810 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


I have successfully executed the code to identify and rank the Product-Store intersections based on their absolute errors. The results have been stored in a new DataFrame with ID `Ranked_Absolute_Errors`, which contains the top intersections along with their absolute error magnitudes and rankings.

Final Answer: The Product-Store intersections have been ranked based on their absolute errors, and the results are available in the DataFrame `Ranked_Absolute_Errors`.

> Finished chain.
{'input': '{\n  "plan": "1. Load the \'Forecasts_Actuals_merged\' dataframe.\\n2. Calculate the forecast error for each Product-Store intersection using the available forecast methods (e.g., forecast_es, forecast_sarima) and actual_sales.\\n3. Identify the intersections with the highest errors.\\n4. Rank these intersections based on their error magnitude to determine which ones to focus on for improvement."\n}', 'output': 'The Product-Store intersections have been ranked based on their absolute errors, and t

2025-12-24 10:13:37,060 - MainProcess - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Final Answer: The intersections to focus on for improving overall accuracy have been identified and ranked based on their absolute errors. You can find the results in the DataFrame `Ranked_Absolute_Errors`. Focus on the intersections with the highest errors, as they have the most potential for improvement.

> Finished chain.
Agent raw response: {'task': 'Find the intersections to focus on to improve overall accuracy. \n              Suggestion should be based only on errors and accuracies. \n              ', 'output': 'The intersections to focus on for improving overall accuracy have been identified and ranked based on their absolute errors. You can find the results in the DataFrame `Ranked_Absolute_Errors`. Focus on the intersections with the highest errors, as they have the most potential for improvement.'}
Total Tokens: 619
Prompt Tokens: 373
Completion Tokens: 246
Total Cost (USD): $0.00020355


In [16]:
df_store.get('Forecast_Accuracy_Analysis')['DataFrame']

,date,avg_absolute_error_forecast_es,avg_absolute_error_forecast_sarima
0,2010-02-05,2.985496e+06,2.855557e+06
1,2010-02-12,2.985496e+06,2.855557e+06
2,2010-02-19,2.985496e+06,2.855557e+06
3,2010-02-26,2.985496e+06,2.855557e+06
4,2010-03-05,2.985496e+06,2.855557e+06
...,...,...,...
163,2013-03-17,2.985496e+06,2.855557e+06
164,2013-03-24,2.985496e+06,2.855557e+06
165,2013-03-31,2.985496e+06,2.855557e+06
166,2013-04-07,2.985496e+06,2.855557e+06


In [25]:
global_code

In [13]:
df_store.get('Accuracy_Store_Product')['DataFrame']

,Store,Product,Accuracy_forecast_es,Accuracy_forecast_sarima,Bias_forecast_es,Bias_forecast_sarima,Absolute_Error_forecast_es,Absolute_Error_forecast_sarima,actual_sales
0,1,1,62.754652,55.701984,7.232931e+05,7.937357e+05,1.199079e+06,1.426133e+06,3219405.18
1,1,97,78.929166,75.143863,9.424863e+05,8.779069e+05,1.060843e+06,1.251420e+06,5034650.88
2,2,2,79.582104,76.286591,1.692303e+06,1.700303e+06,1.924499e+06,2.235119e+06,9425547.92
3,2,16,70.798179,63.362904,7.479398e+05,6.612330e+05,1.127699e+06,1.414830e+06,3861740.69
4,2,81,77.670603,76.056994,9.303381e+05,8.139388e+05,9.898776e+05,1.061410e+06,4433069.25
5,4,9,68.853244,65.834898,1.290896e+06,1.150271e+06,1.533827e+06,1.682466e+06,4924515.45
6,4,94,77.416978,74.253571,1.588002e+06,1.556439e+06,2.201111e+06,2.509441e+06,9746752.24
7,4,96,74.852415,70.747738,1.082837e+06,1.009395e+06,1.378983e+06,1.604065e+06,5483559.95
8,6,72,72.080592,63.651145,2.457435e+06,2.027682e+06,3.201453e+06,4.168038e+06,11466766.90
9,6,93,76.532731,71.469388,1.055840e+06,6.782841e+05,1.387955e+06,1.687423e+06,5914430.12


In [ ]:
df = df_store.get('Accuracy_Store_Product')['DataFrame']

In [16]:
list_dfs_fn({})

'{\n  "status": "ok",\n  "available_dataframes": {\n    "Forecasts_Actuals_merged": {\n      "summary": "DATASET: Results (8400 rows \\u00d7 12 columns) Columns Names: date, actual_sales, Country, Store, Division, Category, Product, forecast_es, forecast_sarima, month, quarter, year\\n\\n",\n      "description": "Dataframe containing Actuals and forecasts (generated at Store,Product) merged. Test/forecast start date :- 2012-04-26. \\n             Predictions during training period also provided."\n    },\n    "Accuracy_Store_Product": {\n      "summary": "DATASET: Results (50 rows \\u00d7 9 columns) Columns Names: Store, Product, Accuracy_forecast_es, Accuracy_forecast_sarima, Bias_forecast_es, Bias_forecast_sarima, Absolute_Error_forecast_es, Absolute_Error_forecast_sarima, actual_sales\\n\\n",\n      "description": "Accuracy numbers stored in DataFrame with id:- Accuracy_Store_Product. Absolute errors calculated at Store-Product-date and subsequently aggregated at Store-Product with 

In [13]:
df_store.list_ids()

['Forecasts_Actuals_merged']

In [15]:
def code_executor_tool_fn(tool_input) -> str:
    
    """
    Executes python code stored in the global variable 'global_code' on the DataFrame referenced by df_id.
    Args: A single JSON string with keys:
        df_id (str): The ID of the DataFrame to use
        timeout (int, optional): Timeout in seconds. Defaults to 9.
    """
    global global_code
    if isinstance(tool_input, str) and tool_input.strip().startswith("{"):
        payload = json.loads(tool_input)
        # code = payload["code"]
        df_id = payload["df_id"]
        timeout = payload.get("timeout", 29)
    try:
        df = df_store.get(df_id)['DataFrame']
    except Exception as e:
        return json.dumps({"status": "error", "error": f"df_id not found: {e}"})

    timeout = int(timeout)
    
    res = run_generated_code_in_subprocess(global_code, df, timeout=timeout)
    print(res)
    if res.get("status") == "error":
        return json.dumps({
            "status": "error",
            "error": res.get("error"),
            "traceback": res.get("traceback"),
            "debug_suggestion": (
                "You can retry code generation using this traceback context "
                "to debug the previous code failure."
            )
        })
    elif res.get("status").startswith("ok"):
        if res.get("status") == "ok_df_generated":
            # result_df: pd.DataFrame = out["df"]
            # store
            
            new_df_id = res.get("df_id") 
            result_dict = res.get("result")
            if new_df_id:
                new_df_id = df_store.put(result_dict['DataFrame'], id=new_df_id)
            else:
                new_df_id = df_store.put(result_dict['DataFrame']) 
            return json.dumps({"status": "ok",
                            "result": f"Executed code and created df_id={new_df_id} with summary {df_store.get(new_df_id)['Summary']}"})
        elif res.get("status") == "ok_str_generated":
            return json.dumps({"status": "ok_non_df", "result": res.get("result")})
        else:
            return json.dumps({"status": "unidentified", "result": res.get("result"), 
                            #    "traceback": res.get("traceback")
                            })
code_executor_tool_fn('{"df_id":"Forecasts_Actuals_merged"}')

{'status': 'error', 'error': 'Execution timed out.', 'traceback': None}


'{"status": "error", "error": "Execution timed out.", "traceback": null, "debug_suggestion": "You can retry code generation using this traceback context to debug the previous code failure."}'

In [12]:
global_code

'```python\ndf = dfs["Forecasts_Actuals_merged"]\n\ngroup_cols = ["Country", "Store", "Division", "Category", "Product"]\n\n# Aggregate actual_sales by the intersection columns\nagg_sales = df.groupby(group_cols, as_index=False)["actual_sales"].sum()\n\n# Get top 5 intersections by total actual_sales\ntop5 = agg_sales.nlargest(5, "actual_sales")\n\n# Merge to filter original df to only include rows from these top 5 intersections\nfiltered_df = df.merge(top5[group_cols], on=group_cols, how="inner")\n\nresult = {"DataFrame": filtered_df, "df_id": "Filtered_Top5_Intersections"}\n```'

In [14]:
df_store.get('Accuracy_intersection')['DataFrame']#.sort_values('Accuracy')

,intersection,Accuracy_forecast_es,Accuracy_forecast_sarima
0,10_40,80.053540,76.479607
1,10_5,68.471419,56.543155
2,10_72,68.879352,61.488059
3,11_46,76.586672,71.367326
4,11_93,76.212120,71.329082
5,12_34,65.940185,65.036902
6,13_13,80.167908,76.772676
7,14_2,78.741453,75.182111
8,15_72,68.206842,54.857609
9,16_40,79.555061,74.248021


In [ ]:
 "The countries with the highest forecast accuracy (lowest WMAPE) are:\n- UK: 76.93\n- USA: 77.05\n- France: 77.15\n- Canada: 77.66\n- Germany: 78.03"

In [20]:

print("\nPer-call token usage:")
for i, tokens in enumerate(token_counter.token_counts, 1):
    print(f"LLM Call {i}: {tokens}")



Per-call token usage:


In [17]:
token_counter.token_counts

[]

In [8]:
from langchain.callbacks import get_openai_callback
user_query = ("Find the Categories with highest Accuracy")
print("USER QUERY:", user_query)

with get_openai_callback() as cb:
    # The agent will call tools according to its ReAct loop
    result = agent_executor.invoke({"input": user_query})
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

USER QUERY: Find the Categories with highest Accuracy


> Entering new AgentExecutor chain...


2025-10-27 13:48:41,680 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


Thought: Need to identify which DataFrame contains category-level forecast accuracy, then compute accuracy per category and find those with the highest values.
Action: list_dfs_fn
Action Input: {}{
  "status": "ok",
  "available_dataframes": {
    "Forecasts_Actuals_merged": {
      "summary": "DATASET: Results (16800 rows \u00d7 6 columns)\nNumeric cols: ['Category', 'forecast', 'actual_sales']\nCategorical cols: ['intersection', 'model', 'date']"
    }
  }
}

2025-10-27 13:48:44,083 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


Thought: Need to calculate accuracy per 'Category' using WMAPE, then identify which categories have the highest accuracy.
Action: calculate_accuracy_fn
Action Input: {"df_id": "Forecasts_Actuals_merged", "forecast_col": "forecast", "actuals_col": "actual_sales", "date_col": "date", "error_calculation_level": ["Category","date"], "error_aggregation_level": ["Category"]}{"status": "ok", "df_id": "Accuracy_Category", "description": "Accuracy numbers stored in DataFrame with id:- Accuracy_Category. Absolute errors calculated at Category-date and subsequently aggregated at Category"}

...

{"status": "ok", "code": "```python\nresult = df.sort_values(by='Accuracy', ascending=False).apply(lambda row: f\"{row['Category']}: {row['Accuracy']}\", axis=1).tolist()\n```"}

2025-10-27 13:48:54,978 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


Action: code_executor_tool_fn
Action Input: {"code": "result = df.sort_values(by='Accuracy', ascending=False).apply(lambda row: f\"{row['Category']}: {row['Accuracy']}\", axis=1).tolist()", "df_id": "Accuracy_Category"}Output intermediate:- {'status': 'ok_random', 'result': "['3.0: 2.5291794020450737', '2.0: 0.25494799004925994', '5.0: -0.24109565128226507', '6.0: -0.560839109898069', '7.0: -1.497860881357525', '1.0: -2.484820315149875', '4.0: -4.040138845978618']"}
{"status": "unidentified", "result": "['3.0: 2.5291794020450737', '2.0: 0.25494799004925994', '5.0: -0.24109565128226507', '6.0: -0.560839109898069', '7.0: -1.497860881357525', '1.0: -2.484820315149875', '4.0: -4.040138845978618']"}

2025-10-27 13:49:03,444 - MainProcess - INFO - HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


Final Answer: The categories with the highest accuracy are **Category 3.0** (accuracy: 2.53) and **Category 2.0** (accuracy: 0.25), as they have the highest accuracy values among all categories in the dataset.

> Finished chain.
Total Tokens: 5608
Prompt Tokens: 5250
Completion Tokens: 358
Total Cost (USD): $0.0


In [13]:
df_store.get('Accuracy_Category')['DataFrame'].sort_values('Accuracy', ascending=False)

,Category,Accuracy
2,3,2.529179
1,2,0.254948
4,5,-0.241096
5,6,-0.560839
6,7,-1.497861
0,1,-2.484820
3,4,-4.040139


In [61]:
top5 = df_store.get('accuracy_Forecasts_Actuals_merged')['DataFrame'].groupby('intersection')['Accuracy'].mean().nlargest(5)
result = "Top 5 intersections with highest WMAPE (worst accuracy): \\\" + \\\", \\\".join(top5.index)

SyntaxError: unterminated string literal (detected at line 2) (3742670631.py, line 2)

In [60]:
top5

intersection
31_13    99.560548
32_92    99.530906
30_92    99.496200
30_91    99.457500
20_90    99.445072
Name: Accuracy, dtype: float64

In [48]:
from langchain_core.language_models.chat_models import BaseChatModel

class GemmaChatLocalHFWithTools(GemmaChatLocalHF):
    def bind_tools(self, tools, tool_choice=None, **kwargs):
        # Just return self, ignoring tool binding for now
        # LangGraph will still handle tool usage externally
        return self

class ChatPerplexityWithTools(ChatPerplexity):
    def bind_tools(self, tools, tool_choice=None, **kwargs):
        # Just return self, ignoring tool binding for now
        # LangGraph will still handle tool usage externally
        return self


In [ ]:
"""
Requirements (pip):
  pip install langchain openai pandas python-dotenv
  # or whichever LLM provider package you use (e.g. openai, anthropic, azure etc.)

Notes:
- Set environment variables for your LLM provider (OPENAI_API_KEY or equivalent).
- This example uses ChatOpenAI from langchain.chat_models as a placeholder; swap to your provider.
"""

import json
import uuid
import traceback
from typing import Dict, Any

import pandas as pd
from langchain.chat_models import ChatOpenAI   # or your LLM class
from langchain.tools import Tool
from langchain.agents import AgentExecutor

import sys, os
import pandas as pd
# sys.path.append(r"c:\Users\Preet Lodaya\Valiance_EDA")
from langchain.agents import initialize_agent, AgentType
# from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage

import ast
from langchain_google_vertexai import GemmaLocalHF, GemmaChatLocalHF

from langchain.agents import AgentExecutor
from langgraph.prebuilt import create_react_agent
from langchain import hub
from langchain.tools import Tool as LangTool
from langchain.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# -----------------------------
# 1) Simple in-memory DataFrame store
# -----------------------------

class DataFrameStore:
    def __init__(self):
        self._store: Dict[str, pd.DataFrame] = {}

    def put(self, df: pd.DataFrame, prefix: str = None, id: str = None) -> str:
        """
        Store a DataFrame with optional prefix and id.
        
        Args:
            df: DataFrame to store
            prefix: Optional string to prepend to the generated id
            id: Optional specific id to use instead of generating one
            
        Returns:
            str: The id used to store the DataFrame
        """
        if id is not None:
            df_id = id
        else:
            df_id = str(uuid.uuid4())
            if prefix is not None:
                df_id = f"{prefix}_{df_id}"
                
        self._store[df_id] = {'DataFrame':df.copy(),
                              'Summary':df_summary(df)}
        return df_id

    def get(self, df_id: str) -> pd.DataFrame:
        return self._store[df_id].copy()

    def list_ids(self):
        return list(self._store.keys())

df_store = DataFrameStore()
df_store.put(results, id = 'Forecasts_Actuals_merged')
# -----------------------------
# 2) Accuracy tool
#    - Accepts a JSON string with keys:
#      { "csv_path": "...", OR "csv_text": "...",
#        "columns": ["pred_col", "actual_col"] (optional),
#        "agg_level": ["store", "region"] (optional)
#      }
#    - Returns JSON: {"status": "ok", "summary": {...}, "df_id": "xxx"}
# -----------------------------

@tool
def list_dfs_fn():  
    '''
    Returns list of available DataFrame ids and their summaries from the store.
    '''
    df_info = {}
    for df_id, content in df_store._store.items():
        df_info[df_id] = {
            "summary": content["Summary"]
        }
    
    return json.dumps({
        "status": "ok", 
        "available_dataframes": df_info
    }, indent=2)

@tool
def calculate_accuracy_fn(df_id: str = Field(description="The ID of the DataFrame to use"),
                    forecast_col: str = Field(description="The name of the forecast column"),
                    actuals_col: str = Field(description="The name of the actuals column"),
                    **kwargs):
    """
   Calculates accuracy of generated forecasts using Weighted MAPE (WMAPE) metric. 
    expects both actual and prediction values to be present in same dataframe. 
    The accuracy is calculated individually for each intersection or combination.
    Output:- Saves a pandas.Series of Accuracy per 'intersection' in the df_store or an error message
    """
    # from ui.ai_analyst_tab import ToolState
    # df = pd.read_csv(r"C:\Users\Preet Lodaya\MLOps_Bot_temp\Results\Stat_forecast_20120426.csv") #ToolState.df
    df = df_store.get(df_id)
    if df is None:
        return "No dataframe available"
    if forecast_col is None or actuals_col is None:
        return "forecast_col and actuals_col required"

    try:
        grouped = df.groupby('intersection').apply(
            lambda x: np.sum(np.abs(x[forecast_col] - x[actuals_col])) / np.sum(x[actuals_col]) * 100
        )
        description = f"WMAPE calculated between {forecast_col} and {actuals_col} for dataframe with id {df_id}"
        if kwargs['agg'] is not None:
            agg_levels = [v for k,v in kwargs['agg'].items()]
            grouped_id = df_store.put(grouped, id = f"Accuracy_{'_'.join(agg_levels)}")
            description += f", aggregated by {', '.join(agg_levels)}"
        else:
            grouped_id = df_store.put(grouped, id = f"Accuracy_{df_id}")
        # return grouped
        return json.dumps({
            "status": "ok",
            "df_id": grouped_id,
            "description": description
        })

    except Exception as e:
        # return f"WMAPE calculation error: {str(e)}"
        return json.dumps({
            "status": "Error",
            "description": str(e)
        })

# -----------------------------
# 3) Code generator tool (simple wrapper that calls an LLM to produce python code only)
#    - expects input JSON: {"goal": "what to do with df", "df_id": "..."}
#    - returns JSON: {"status":"ok","code":"<python code as string>"}
# -----------------------------
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# create a small prompting function: ask LLM to return ONLY a python function named 'transform(df)'
# that accepts `df` (pandas) and returns either a dataframe or prints results.

# code_generator: ask LLM to produce python code ONLY implementing transform(df)
code_prompt_template = PromptTemplate(
    input_variables=["goal", "df_id", "context_cols"],
    template=(
        "You are a python coding assistant. Produce ONLY Python code (no explanation) that "
        "implements a function `def transform(df):` which will be run with a pandas DataFrame `df`.\n\n"
        "Goal: {goal}\n"
        "DataFrame id: {df_id}\n"
        "Columns available: {context_cols}\n\n"
        "Return a pandas DataFrame or a serializable object. Do not import modules; pandas is available as `pd`.\n"
        "Be defensive: check for missing columns. Only emit python code."
    )
)
# code_llm = GemmaChatLocalHF(model="google/gemma-3-1b-it", temperature=0.01, streaming=True, hf_access_token=HF_KEY)#ChatPerplexity(model="sonar", temperature=0.01, streaming=True)
code_llm =  ChatPerplexity(model="sonar", temperature=0.01, streaming=True)
# code_chain = LLMChain(llm=code_llm, prompt=code_prompt_template)
code_chain = code_prompt_template | code_llm 
@tool
def code_generator_tool_fn(goal: str = Field(description="What the code should achieve"), 
                           df_id: str = Field(description="The ID of the DataFrame to use")) -> str:
    '''
    Generates python code to apply transformation on pandas dataframes. 
    Returns: 
        string: Python code.
    '''
    try:
        sample_df = df_store.get(df_id)
    except Exception as e:
        return json.dumps({"status": "error", "error": f"df_id not found: {e}"})

    context_cols = ", ".join(list(sample_df.columns[:50]))
    code = code_chain.run(goal=goal, df_id=df_id, context_cols=context_cols)
    return json.dumps({"status": "ok", "code": code})

# code_executor: runs generated code safely and stores result
@tool
def code_executor_tool_fn(
    code: str = Field(description="Python code to execute"), 
    df_id: str = Field(description="The ID of the DataFrame to use"),
    timeout: int = Field(default=8, description="Timeout in seconds")
) -> str:
    """
    Executes generated python code on the DataFrame referenced by df_id.
    """
    try:
        df = df_store.get(df_id)
    except Exception as e:
        return json.dumps({"status": "error", "error": f"df_id not found: {e}"})

    timeout = int(timeout)

    res = run_generated_code_in_subprocess(code, df, timeout=timeout)
    if res.get("status") == "ok":
        return json.dumps({"status": "ok", "new_df_id": res["df_id"], "meta": res["meta"],
                           "observation": f"Executed code and created df_id={res['df_id']} (shape={res['meta']['shape']})"})
    elif res.get("status") == "ok_non_df":
        return json.dumps({"status": "ok_non_df", "result": res.get("result")})
    else:
        return json.dumps({"status": "error", "error": res.get("error"), "traceback": res.get("traceback")})

# -----------------------------
# 4) Wrap both as LangChain Tools
# -----------------------------
accuracy_tool = Tool(
    name="accuracy_tool",
    func=calculate_WMAPE,
    description=(
        "Compute accuracy between predicted and actual columns in a CSV. "
        "Input: JSON string with keys: csv_path or csv_text, optional 'columns': [pred_col, actual_col], optional 'agg_level'. "
        "Returns JSON with status and 'df_id' referencing the computed DataFrame."
    ),
)

code_generator_tool = Tool(
    name="code_generator",
    func=code_generator_tool_fn,
    description="Generate Python code (ONLY code) to transform a DataFrame referenced by df_id. Input JSON: {'goal':..., 'df_id':...}."
)

code_executor_tool = Tool(
    name="code_executor",
    func=code_executor_tool_fn,
    description="Safely execute generated python code on the DataFrame referenced by df_id. Input JSON: {'code':..., 'df_id':..., 'timeout': optional}."
)


STRICT_REACT_PROMPT = """You are an analytical agent that evaluates DataFrames.

IMPORTANT: You MUST follow this format EXACTLY. Do NOT deviate.

When you need to take an action:
Thought: <brief reasoning>
Action: <tool_name>
Action Input: <JSON>

When you receive a result:
Observation: <the result>

Then either:
- Continue with another Thought/Action/Observation cycle, OR
- When done, respond ONLY with:

Final Answer: <your answer here - no markdown headers, no bullet points, just plain text>

CRITICAL RULES:
- Never include <think> tags
- Never mix Final Answer with actions
- No markdown formatting in Final Answer
- No headers or bullet points in Final Answer
- Keep responses concise
- Only one Thought/Action/Observation per response cycle
- If the function requires no input, provide an empty JSON object

"""
# System prompt
system_prompt = SystemMessage(
    content=(STRICT_REACT_PROMPT
    )
)
            
# -----------------------------
# 5) Create the ReAct agent and AgentExecutor
# -----------------------------
# Primary agent LLM (reasoning/orchestration). Use a non-code model if you like.
# orchestrator_llm = GemmaChatLocalHFWithTools(
#     model="google/gemma-3-1b-it",
#     temperature=0.01,
#     streaming=True,
#     hf_access_token=HF_KEY,
# )
orchestrator_llm =  ChatPerplexityWithTools(model="sonar-pro", temperature=0.01, streaming=True)
# Create the react agent runnable
# Note: create_react_agent returns a Runnable (per docs). We pass the llm, tools and a default prompt if desired.
react_agent = create_react_agent(
    model=orchestrator_llm,
    prompt=system_prompt,
    tools=[calculate_accuracy_fn, code_generator_tool_fn,code_executor_tool_fn, list_dfs_fn],
    verbose=True
    # You could pass a custom prompt template here. For most cases defaults work.
)

# # Build the executor that will run the agent and call tools
# agent_executor = AgentExecutor.from_agent_and_tools(
#     agent=react_agent,
#     tools=[accuracy_tool, code_generator_tool,code_executor_tool],
#     verbose=True,
#     max_iterations=6,                # limit loops so it doesn't go forever
#     handle_parsing_errors=True,
# )

# -----------------------------
# 6) Example orchestration flow:
#    - user asks: "Evaluate accuracy on file X and produce a per-store accuracy table, then create a 'flag' column for low-accuracy stores."
#    - the ReAct agent will (ideally):
#         1) call accuracy_tool with csv_path or csv_text
#         2) receive df_id
#         3) decide it needs a custom transformation -> call code_generator with df_id and a clear goal
#         4) receive code, then (outside the agent) we validate and execute the code on the DataFrame referenced by df_id
# -----------------------------
def run_example():
    user_query = ("Calculate accuracy for the full scope of forecasts"
    ) 
    print("USER QUERY:", user_query)

    # The agent will call tools according to its ReAct loop
    result = react_agent.invoke({"messages": [HumanMessage(content=user_query)]})
    print("Agent raw response:", result)
    return result
    # NOTE: The agent will have called the accuracy tool and (if it wants) the code generator.
    # The actual code execution on df_id is done here by the system: we look for a recent df_id in the store and see if the agent
    # appended an instruction to run code. This is a simple post-processing demonstration.

if __name__ == "__main__":
    out = run_example()


TypeError: create_react_agent() got unexpected keyword arguments: {'verbose': True}

In [ ]:
out['']

{'messages': [HumanMessage(content='Calculate wmape', additional_kwargs={}, response_metadata={}, id='7bae7a3e-7b65-4a0b-be7d-be28cc9afb32'),
  AIMessage(content='To calculate WMAPE (Weighted Mean Absolute Percentage Error), you need actual values and forecasted values from your dataset. The formula is:\n\n\\[ \\text{WMAPE} = \\frac{\\sum_{i=1}^{n}|A_i - F_i|}{\\sum_{i=1}^{n}|A_i|} \\]\n\nwhere \\(A_i\\) represents actual values and \\(F_i\\) represents forecasted values[3].\n\n## Calculation Steps\n\n**1. Calculate the absolute errors**\nFor each data point, find \\(|A_i - F_i|\\), which is the absolute difference between actual and forecasted values[2].\n\n**2. Sum the absolute errors**\nAdd up all the absolute errors from step 1 to get \\(\\sum_{i=1}^{n}|A_i - F_i|\\)[7].\n\n**3. Sum the actual values**\nCalculate the total of all actual values: \\(\\sum_{i=1}^{n}|A_i|\\)[5].\n\n**4. Divide to get WMAPE**\nDivide the sum of absolute errors by the sum of actual values. Multiply by 10

In [29]:
print(calculate_WMAPE.args)

{'df_id': {'title': 'Df Id', 'type': 'string'}, 'forecast_col': {'title': 'Forecast Col', 'type': 'string'}, 'actuals_col': {'title': 'Actuals Col', 'type': 'string'}, 'kwargs': {'additionalProperties': True, 'default': None, 'title': 'Kwargs', 'type': 'object'}}


In [ ]:
from pydantic import BaseModel, Field
